For each NACE Class get the 100 chunks that scored highest across all the reports 

In [76]:
import pandas as pd
import glob
import os
import tqdm
import numpy as np
import matplotlib.pyplot as plt
from test_base import *

In [77]:
overview_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/datasets/stoxx_600/stoxx_600_overview.csv"
overview_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/datasets/reports_subset_from_full_data_1/reports_subset_from_full_data_1_overview.csv"

In [78]:
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/paragraph_and_sentence_len_3_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/paragraph_and_sentence_len_3_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/paragraph_and_sentence_len_3_min_chunk_len_0_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/tables_cos_sim_0.0_nace_level_1_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/sentence_len_5/paragraph_and_sentence_len_5_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/sentence_len_5/paragraph_and_sentence_len_5_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/paragraph_and_sentence_len_6_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/"

In [79]:
reports = glob.glob(raw_data_path + "*/*_long.csv")
reports = glob.glob(raw_data_path + "*/*_short.csv")

In [80]:
sample_ratio = 1

In [81]:
max_elements_per_class = 1000000

top_k_sentences = 200000

In [82]:
# if true, adds only chunks to training that have been classified into the same NACE class its report comes from
filter_only_right_chunks = True

In [83]:
# if true, adds random chunks that do not fulfill the minimum treshold (for BERT Training)
with_null_classifiers = False

In [84]:
new_threshold_cos_sin = 0.4

In [85]:
nace_level_descriptions = 2
nace_level = 1
assert nace_level_descriptions >= nace_level

In [86]:
training_data_path = "../data/training_data/"

In [87]:
suffix = f"sample_ratio_{sample_ratio}" + ("__filter_only_right_chunks" if filter_only_right_chunks else "") + ("__with_null_classifiers" if with_null_classifiers else "") + f"__nace_level_{nace_level}"

# "_subsample" if sample_ratio != 1 else ""
# end_path = os.path.join("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/test_read_training_data_right_classifications", raw_data_path.split("/")[-2] + suffix)
# end_path = os.path.join("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/test_read_training_data", raw_data_path.split("/")[-2] + suffix + "_with_no_class__cos_sim_04")
# end_path = os.path.join("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/test_read_training_data_right_classifications", raw_data_path.split("/")[-2] + suffix + "_with_no_class__cos_sim_04")

end_path = os.path.join(training_data_path, 
                        raw_data_path.split("/")[-2] + "__" + suffix)
end_path

'../data/training_data/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2__sample_ratio_1__filter_only_right_chunks__nace_level_1'

In [88]:
#df_overview = pd.read_excel(overview_path)
df_overview = pd.read_csv(overview_path)
df_overview

,Unnamed: 0,Symbol,Name,Company is Active,Company Founded Date,Country of Primary Listing Iso3,CUSIP,Date Of First Trade,Entity Country HQ,Entity Credit Parent,...,ISIN,Primary Equity Listing,Proper Name,Public Company,Region Ticker,Sec is Primary Issue,Sec Type,SEDOL,NACE_letter,Report
0,5789,ZW0009011041,Ariston Holdings Ltd.,1,1947.0,ZWE,V97772103,20090317.0,ZWE,@NA,...,ZW0009011041,603408,Ariston Holdings Ltd.,1.0,ARIS-ZW,1,SHARE,6034081,A,Ariston Holdings Ltd.1.pdf
1,35816,INE978A01027,Heritage Foods Limited,1,1992.0,IND,Y3179H146,20020117.0,IND,06FQLY-E,...,INE978A01027,BF2F40,Heritage Foods Limited,1.0,519552-IN,1,SHARE,BF2F405,A,Heritage Foods Limited1.pdf
2,80373,MYL7854OO002,Timberwell Bhd.,1,1996.0,MYS,Y88399103,19970516.0,MYS,05JH15-E,...,MYL7854OO002,690556,Timberwell Bhd.,1.0,7854-MY,1,SHARE,6905563,A,Timberwell Bhd.1.pdf
3,49813,MYQ0189OO009,Matang Bhd.,1,2015.0,MYS,Y58347108,20170117.0,MYS,@NA,...,MYQ0189OO009,BYYQB5,Matang Bhd.,1.0,0189-MY,1,SHARE,BYYQB53,A,Matang Bhd.2.pdf
4,73064,MYL4316OO005,Sin Heng Chan (Malaya) Bhd.,1,1962.0,MYS,Y80178109,19880324.0,MYS,05YMQ5-E,...,MYL4316OO005,681088,Sin Heng Chan (Malaya) Bhd.,1.0,4316-MY,1,SHARE,6810883,A,Sin Heng Chan (Malaya) Bhd.1.pdf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1557,5783,US0404432025,Aristocrat Group Corp.,0,2011.0,USA,040443202,20120313.0,USA,00DDP9-E,...,US0404432025,04044320,Aristocrat Group Corp.,1.0,ASCC-US,1,SHARE,BWX6257,S,Aristocrat Group Corp.1.pdf
1558,73228,KYG816BW1095,Sino-Life Group Limited,1,2005.0,HKG,G816BW109,20090909.0,HKG,00C60C-E,...,KYG816BW1095,B409GR,Sino-Life Group Limited,1.0,8296-HK,1,SHARE,B409GR3,S,Sino-Life Group Limited1.pdf
1559,86319,KYG9477E1070,Water Oasis Group Limited,1,1998.0,HKG,G9477E107,20070905.0,HKG,00610B-E,...,KYG9477E1070,651233,Water Oasis Group Limited,1.0,WOSSF-US,0,SHARE,B02V9Q9,S,Water Oasis Group Limited1.pdf
1560,66027,US76119X1054,"Reservoir Media, Inc.",1,2007.0,USA,76119X105,20210105.0,USA,0NVYZ2-E,...,US76119X1054,76119X10,"Reservoir Media, Inc.",1.0,RSVR-US,1,SHARE,BP0B9H7,S,"Reservoir Media, Inc.1.pdf"


In [89]:
df_nace_codes_descriptions = pd.read_csv("../data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv", sep="\t")
filter_level_1_classes = "ABCDEFGHIJKLMNOPQRSTUVW"

In [90]:
reports

['/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/PVH Corp.3.txt/PVH Corp.3.txt_short.csv',
 '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/Mekdam Holding Group Company1.txt/Mekdam Holding Group Company1.txt_short.csv',
 '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/Ambea AB1.txt/Ambea AB1.txt_short.csv',
 '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/Harbour Equine Holdings Limited3.txt/Harbour E

In [109]:
result = pd.DataFrame(columns=["Sentences", "Score", "NACE_Code"])

# loop over all reports and get the all the sentences 
for report in tqdm.tqdm(reports):

    if "Four Seasons Education" not in report: 
        continue
    df = pd.read_csv(report)

    if filter_only_right_chunks: 
        report_name = os.path.basename(report).replace(".txt_short.csv", "") + ".pdf"
        report_code = df_overview[df_overview["Report"]==report_name]["NACE"].iloc[0]
        report_code = get_all_level(report_code)[nace_level]
        if nace_level == nace_level_descriptions: 
            filter_column = list(filter(lambda x: "Scores_"+str(report_code) in x, df.columns))
        else: 
    
            filter_column = []
            for column in df.columns: 
                if "Scores" in column: 
                    if get_all_level(column.split("_")[1])[nace_level]==report_code: 
                        filter_column.append(column)
                #print(column.split("_"))

            print(filter_column)

        df = df[["Sentences"]+filter_column]

    if nace_level == 1: 
        # filter some nace classes
        scores = df[[column for column in df.columns if ("Scores" in column) and get_all_level(column.split("_")[1], df_nace_codes_descriptions)[nace_level] in filter_level_1_classes]].columns
    else: 
        scores = df[[column for column in df.columns if ("Scores" in column)]].columns
    
    for score in scores:
        temp = df[df[score].notna()][["Sentences", score]]  
        try: 
            temp["NACE_Code"] = get_all_level(score.split("_")[1])[nace_level]
        except IndexError: 
            continue
        temp = temp.rename(columns={score: "Score"})
        result = pd.concat([result, temp])

  0%|          | 0/1555 [00:00<?, ?it/s]/var/folders/fp/yhl61lbj3m73x3_17tsp1vrr0000gn/T/ipykernel_54910/3357708086.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result = pd.concat([result, temp])
100%|██████████| 1555/1555 [00:00<00:00, 64708.29it/s]

['Scores_85_Education']


In [114]:
result[result["Score"] > 0.2]

,Sentences,Score,NACE_Code
48,all of the other ordinary shares outstanding a...,0.206159,P
162,on september the board of directors authorized...,0.221320,P


In [92]:
os.makedirs(end_path, exist_ok=True)

In [93]:
recordings = []

In [94]:
# for each code, store the 100 with the highest similarity score to the code

full_df = []
for code in set(result["NACE_Code"].to_list()): 

    #if not get_all_level(code.split("_")[1], df_nace_codes_descriptions)[1] in filter_level_1_classes: 
    # if not get_all_level(code, df_nace_codes_descriptions)[nace_level] in filter_level_1_classes: 
    #     continue

    temp = result[result["NACE_Code"] == code]
    temp = temp.drop_duplicates(subset="Sentences")
    temp = temp[temp["Sentences"].apply(len) >= 100]
    temp = temp.sort_values(by="Score", ascending=False)

    if with_null_classifiers: 
        temp.loc[temp["Score"]<new_threshold_cos_sin, "NACE_Code"] = "NO_CLASS"
        
        class_index = temp[temp["NACE_Code"]!="NO_CLASS"].index
        no_class_index = temp[temp["NACE_Code"]=="NO_CLASS"].index

        print(len(temp[temp["NACE_Code"]=="NO_CLASS"]))
        print(temp[temp["NACE_Code"]=="NO_CLASS"].index)
        print(temp.loc[no_class_index])
        print(code)
        print("--")

        temp = temp.loc[list(np.random.choice(no_class_index, len(class_index)))+list(class_index)]
    else:
        temp = temp[temp["Score"] >= new_threshold_cos_sin]

    number_of_elements_per_class = min(int(sample_ratio*len(temp)), max_elements_per_class, len(temp))
    random_choice = np.random.choice(len(temp), number_of_elements_per_class, replace=False)
    temp = temp.iloc[random_choice]
    temp = temp.sort_values(by="Score", ascending=False)
    temp = temp.iloc[:top_k_sentences, :]
    temp = temp.reset_index(drop=True)
    temp["Evaluation"] = None
    temp["Notes"] = None
    temp = temp[["Evaluation", "Notes", "Sentences", "Score", "NACE_Code"]]

    recordings.append({"Code": code,"Nbr. of Chunks": len(temp),"Avg. Length": temp["Sentences"].apply(len).mean(), "Avg. Score": temp["Score"].mean(), "Min. Score": temp["Score"].min(), "Max. Score": temp["Score"].max()})

    text = ""
    for i, row in temp.iterrows():
        text += f"#{i}, Score: " + str(round(row["Score"], 2)) + "\n\n" + row["Sentences"] + "\n\n"

    with open(os.path.join(end_path, code.replace("/"," ")) + ".txt", "w") as f:
        f.write(text)

    temp.to_csv(os.path.join(end_path, code.replace("/"," ")) + ".csv")

    full_df.append(temp)

full_df = pd.concat(full_df, axis=0, ignore_index=True)

In [95]:
df_recordings = pd.DataFrame(recordings)
df_recordings = df_recordings.sort_values(by="Code")
df_recordings.head()

,Code,Nbr. of Chunks,Avg. Length,Avg. Score,Min. Score,Max. Score
18,A,2273,476.214694,0.449670,0.400010,0.704805
16,B,7582,523.860987,0.458783,0.400019,0.669105
17,C,8274,583.588349,0.441124,0.400010,0.625020
15,D,0,NaN,NaN,NaN,NaN
3,E,2318,509.453840,0.470171,0.400047,0.678239


In [96]:
df_recordings.to_csv(end_path + "/statistics.csv")

In [97]:
full_df= full_df.rename(columns={"Sentences": "text"})
#full_df = full_df.drop(columns="Score")
full_df

,Evaluation,Notes,text,Score,NACE_Code
0,None,None,information planning division information syst...,0.671113,M
1,None,None,gt diverse range and provision of services acr...,0.667651,M
2,None,None,our firstopinion and referral practices provid...,0.662930,M
3,None,None,generally agencies or product and service owne...,0.661716,M
4,None,None,the biological assets are exposed to domestic ...,0.661653,M
...,...,...,...,...,...
85351,None,None,in addition the company faces climaterelated h...,0.400242,A
85352,None,None,selain itu pada ranah kebijakan internasional ...,0.400090,A
85353,None,None,i malaysia the operations in this area are pri...,0.400039,A
85354,None,None,the company has also ceased the use of coal to...,0.400038,A


In [98]:
from sklearn.model_selection import train_test_split

# Split full_df into train (60%) and temp (40%)
train_df, temp_df = train_test_split(full_df, test_size=0.4, random_state=42)

# Split temp into test (20%) and validation (20%)
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Print the sizes of each split
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}, Validation size: {len(val_df)}")

Train size: 51213, Test size: 17071, Validation size: 17072


In [99]:
full_df.to_csv(end_path + "/full_data.csv", index=False)

In [100]:
train_df.to_csv(end_path + "/train_data.csv", index=False)
val_df.to_csv(end_path + "/val_data.csv", index=False)
test_df.to_csv(end_path + "/test_data.csv", index=False)

In [101]:
end_path

'../data/training_data/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2__sample_ratio_1__filter_only_right_chunks__nace_level_1'